# Reproducing the figures and tables of the paper

**Plug-and-play with a decreasing noise level.**  Five PnP / RED algorithms are run twice on every
problem: once with a **constant** noise level $\sigma_c$, once with the **annealed** schedule

$$\sigma_k = \varepsilon + (\sigma_0 - \varepsilon)\, r^k , \qquad 0 < r < 1, \quad \varepsilon > 0 .$$

Every parameter is written in the cell that uses it.  Each problem has one parameter cell with a
line per run: `Params(K, sigma_0=..., r=..., eps=..., lam=...)` for an annealed run,
`Params(K, sigma_c=..., lam=...)` for a constant one.  These are the values selected on the
validation images (see the appendix of the paper).

* the **figures** are made on the image shown in the paper, with the same random seed;
* the **tables** average over the first `N_IMAGES` test images (paper: 100 faces, 68 CBSD68 images).

In [ ]:
import os
import sys

sys.path.insert(0, os.getcwd())          # the modules of this folder

import matplotlib.pyplot as plt
import numpy as np
import torch

import algorithms
import data
import denoisers
import figures
import problems
import runner
import schedules
from params import Params
from utils import get_device, psnr

DEVICE = get_device()
DATA = "data"                            # data/ffhq128, data/CBSD68_png
CKPTS = "checkpoints"                    # the four denoisers
print("device:", DEVICE)

## Parameters common to every experiment

In [ ]:
IMAGE_SIZE = 128            # every experiment runs at 128 x 128
N_IMAGES = 8                # images averaged in the tables (paper: 100 FFHQ faces, 68 CBSD68 images)
RECORD_OBJECTIVE = True     # middle panel of the convergence figures (costs extra denoiser calls)

# Annealed schedule  sigma_k = eps + (sigma_0 - eps) r^k
sigma_0_ffhq = 5.0          # faces
sigma_0_cbsd68 = 1.0        # natural images
r = 0.98                    # geometric ratio, the same everywhere

# Denoisers
delta = 0.7                 # relaxation of the proximal denoisers: D = Id - delta grad g   (PnP algorithms)
L_g_gs_diffunet = 4.2       # Lipschitz constant of grad g, GS-DiffUNet (FFHQ, RED algorithms)
L_g_gs_drunet = 2.2         # Lipschitz constant of grad g, GS-DRUNet   (CBSD68, RED algorithms)
# RED step  gamma = gamma_rel / (L_f + lam L_g) with gamma_rel = 1 (the default of Params);
# PnP step  gamma = c_f / L_f, with c_f given per problem.

## The schedule

$\sigma_k$ decreases geometrically and converges to the floor $\varepsilon$ **without ever reaching it**.

In [ ]:
K = 500
eps = 0.014
sigmas = schedules.annealed_sigmas(K, sigma_0_ffhq, eps, r)
print(f"sigma_0 = {sigmas[0]:.3f}   sigma_K = {sigmas[-1]:.4f}   eps = {eps}")
figures.plot_schedule(sigmas, eps, sigma_c=2.0)
plt.show()

## Loading the faces and the two FFHQ denoisers

In [ ]:
faces = data.ffhq(os.path.join(DATA, "ffhq128"), DEVICE, n=100, size=IMAGE_SIZE)     # the 100 test faces
gs_diffunet = denoisers.gs_diffunet(os.path.join(CKPTS, "gs_diffunet128.pt"), DEVICE)
prox_diffunet = denoisers.prox_diffunet(os.path.join(CKPTS, "prox_diffunet128.pt"), DEVICE, delta=delta)
dpir_denoiser = denoisers.pretrained_drunet(DEVICE)     # the DPIR baseline runs on the plain pretrained DRUNet
print(faces.shape)

## Figure 1 — why a decreasing noise level

RED gradient descent on a $64 \times 64$ hole with the **same denoiser** every time,
$x_{k+1} = x_k - \gamma [\nabla f(x_k) + \lambda (x_k - D_{\sigma_k}(x_k))]$.
A small constant $\sigma$ leaves the initial flat patch in place, a large one fills the hole with a
blurred face; decreasing $\sigma_k$ from $5$ to $0.014$ generates the missing content.
The figure shows the last iterate $x_K$.

In [ ]:
# parameters of Figure 1
image = 12                  # the face shown
K = 400
gamma_rel = 1.9
sigma_0, eps, lam_annealed = 5.0, 0.014, 3.0
constant_runs = [(0.02, 1.0), (0.5, 0.3), (1.0, 0.3)]     # (sigma_c, lam): the best lam at each level

hole = problems.hole_inpainting(IMAGE_SIZE, DEVICE, hole=64, sigma_noise=0.01)
x_true = faces[:16]
y = hole.measure(x_true)
x_0 = hole.init(y)

columns = []
for sigma_c, lam in constant_runs:
    gamma = gamma_rel / (hole.lipschitz + lam * L_g_gs_diffunet)
    x = algorithms.red_gd(y, hole, gs_diffunet, x_0, schedules.constant_sigmas(K, sigma_c), lam, gamma)
    columns.append((f"constant\n$\\sigma$ = {sigma_c:g}", x[image]))
gamma = gamma_rel / (hole.lipschitz + lam_annealed * L_g_gs_diffunet)
x = algorithms.red_gd(y, hole, gs_diffunet, x_0, schedules.annealed_sigmas(K, sigma_0, eps, r), lam_annealed, gamma)
columns.append((f"annealed\n{sigma_0:g} $\\to$ {eps}", x[image]))

figures.restoration_strip(x_true[image], x_0[image], columns)
plt.show()

## Helpers used by every other figure

`run_all` runs the five algorithms in both versions (and DPIR) on a batch of images: the tables.
`paper_figure` runs RED-GD, SNORE, PnP-PGD and SNOPnP on the single image of the paper's figure,
with its seed, records the curves, and draws the restoration strip and the three convergence panels.

In [ ]:
def run_all(problem, images, annealed, constant, red_denoiser, prox_denoiser, L_g):
    # the five algorithms, annealed and constant, plus DPIR, on `images`
    y = problem.measure(images)
    results = {}
    for version, runs in (("annealed", annealed), ("constant", constant)):
        for algorithm, p in runs.items():
            denoiser = prox_denoiser if algorithm in runner.PNP_ALGORITHMS else red_denoiser
            results[(algorithm, version)] = result = runner.run(algorithm, problem, y, images, denoiser, p, L_g)
            print(f"{runner.LABELS[algorithm]:8s} {version:9s} PSNR {result.psnr.mean():6.2f} dB")
    results["dpir"] = runner.run_dpir(problem, y, images, dpir_denoiser)
    print(f"{'DPIR':8s} {'':9s} PSNR {results['dpir'].psnr.mean():6.2f} dB")
    return results


def paper_figure(problem, images, index, annealed, constant, red_denoiser, prox_denoiser, L_g,
                 zoom=None, psnr_floor=None):
    # the restoration strip and the convergence curves of image `index`
    y_all = problem.measure(images)                 # the measurements of the whole test set, then one image
    x_true, y = images[index:index + 1], y_all[index:index + 1]
    shown = ["red_gd", "snore", "pnp_pgd"]          # in the restoration strip
    curves = ["red_gd", "snore", "pnp_pgd", "snopnp"]
    results = {}
    for version, runs in (("annealed", annealed), ("constant", constant)):
        for algorithm in curves:
            denoiser = prox_denoiser if algorithm in runner.PNP_ALGORITHMS else red_denoiser
            results[(algorithm, version)] = runner.run(algorithm, problem, y, x_true, denoiser, runs[algorithm], L_g,
                                                       seed=index, record_objective=RECORD_OBJECTIVE and version == "annealed")
    dpir = runner.run_dpir(problem, y, x_true, dpir_denoiser)
    columns = [("DPIR", dpir.estimate[0])]
    for algorithm in shown:
        for version in ("constant", "annealed"):
            columns.append((f"{runner.LABELS[algorithm]}\n{version} $\\sigma$", results[(algorithm, version)].estimate[0]))
    if zoom == "auto":                              # where the constant RED-GD run has the largest colour error
        zoom = figures.worst_window(results[("red_gd", "constant")].estimate[0], x_true[0], 40)
    figures.restoration_strip(x_true[0], problem.init(y)[0], columns, zoom=zoom)
    figures.plot_curves({a: results[(a, "annealed")] for a in curves},
                        {a: results[(a, "constant")] for a in curves}, psnr_floor=psnr_floor)
    plt.show()
    return results

## Figure 2 — FFHQ, inpainting of a $64 \times 64$ hole

In [ ]:
# parameters: one line per run
K = 500
sigma_0 = sigma_0_ffhq
r = 0.98

hole_annealed = {
    "red_gd":  Params(K, sigma_0=sigma_0, r=r, eps=0.014, lam=0.3),
    "pnp_pgd": Params(K, sigma_0=sigma_0, r=r, eps=0.01, c_f=0.3),
    "snore":   Params(K, sigma_0=sigma_0, r=r, eps=0.01, lam=0.3, q=0.55),
    "snopnp":  Params(K, sigma_0=sigma_0, r=r, eps=0.01, c_f=0.95, q=1, warm_mult=1),
    "ered":    Params(K, sigma_0=sigma_0, r=r, eps=0.01, lam=0.5, q=1, warm_mult=1, group="hflip"),
}
hole_constant = {
    "red_gd":  Params(K, sigma_c=2, lam=0.05),
    "pnp_pgd": Params(K, sigma_c=0.04, c_f=0.95),
    "snore":   Params(K, sigma_c=2, lam=0.1, q=0.55),
    "snopnp":  Params(K, sigma_c=0.16, c_f=0.95, q=0.55),
    "ered":    Params(K, sigma_c=2, lam=0.1, q=0.55, group="hflip"),
}

In [ ]:
hole = problems.hole_inpainting(IMAGE_SIZE, DEVICE, hole=64, sigma_noise=0.01)
_ = paper_figure(hole, faces, 50, hole_annealed, hole_constant, gs_diffunet, prox_diffunet, L_g_gs_diffunet)

## Figure 3 — FFHQ, demosaicing (Bayer pattern)

The artefacts are colour fringes along the edges; the second row zooms on the $40 \times 40$ window
where the constant RED-GD run has the largest colour error.

In [ ]:
# parameters: one line per run
K = 500
sigma_0 = sigma_0_ffhq
r = 0.98

demosaic_annealed = {
    "red_gd":  Params(K, sigma_0=sigma_0, r=r, eps=0.01, lam=0.1),
    "pnp_pgd": Params(K, sigma_0=sigma_0, r=r, eps=0.01, c_f=0.5),
    "snore":   Params(K, sigma_0=sigma_0, r=r, eps=0.01, lam=0.3, q=0.55, warm_mult=1),
    "snopnp":  Params(K, sigma_0=sigma_0, r=r, eps=0.01, c_f=0.5, q=1),
    "ered":    Params(K, sigma_0=sigma_0, r=r, eps=0.01, lam=1, q=1, warm_mult=1, group="hflip"),
}
demosaic_constant = {
    "red_gd":  Params(K, sigma_c=0.08, lam=0.2),
    "pnp_pgd": Params(K, sigma_c=0.04, c_f=0.95),
    "snore":   Params(K, sigma_c=0.04, lam=0.6, q=0.55),
    "snopnp":  Params(K, sigma_c=0.028, c_f=0.95, q=0.55),
    "ered":    Params(K, sigma_c=0.08, lam=0.2, q=0.55, group="hflip"),
}

In [ ]:
demosaic = problems.demosaicing(IMAGE_SIZE, DEVICE, sigma_noise=0.01)
_ = paper_figure(demosaic, faces, 22, demosaic_annealed, demosaic_constant, gs_diffunet, prox_diffunet,
                 L_g_gs_diffunet, zoom="auto")

## Figure 4 — FFHQ, super-resolution $\times 4$

In [ ]:
# parameters: one line per run
K = 500
sigma_0 = sigma_0_ffhq
r = 0.98

sr_ffhq_annealed = {
    "red_gd":  Params(K, sigma_0=sigma_0, r=r, eps=0.08, lam=0.01),
    "pnp_pgd": Params(K, sigma_0=sigma_0, r=r, eps=0.04, c_f=0.3),
    "snore":   Params(K, sigma_0=sigma_0, r=r, eps=0.056, lam=0.01, q=1, warm_mult=1),
    "snopnp":  Params(K, sigma_0=sigma_0, r=r, eps=0.08, c_f=0.95, q=0.7),
    "ered":    Params(K, sigma_0=sigma_0, r=r, eps=0.08, lam=0.01, q=1, warm_mult=1, group="hflip"),
}
sr_ffhq_constant = {
    "red_gd":  Params(K, sigma_c=0.2, lam=0.01),
    "pnp_pgd": Params(K, sigma_c=0.0571, c_f=0.15),
    "snore":   Params(K, sigma_c=0.112, lam=0.02, q=0.55),
    "snopnp":  Params(K, sigma_c=0.08, c_f=0.7, q=0.7),
    "ered":    Params(K, sigma_c=0.2, lam=0.01, q=0.55, group="hflip"),
}

In [ ]:
sr_ffhq = problems.super_resolution(IMAGE_SIZE, DEVICE, factor=4, sigma_noise=0.01)
_ = paper_figure(sr_ffhq, faces, 97, sr_ffhq_annealed, sr_ffhq_constant, gs_diffunet, prox_diffunet,
                 L_g_gs_diffunet, zoom=(38, 28, 48), psnr_floor=23)

## Table 1 — FFHQ

Mean PSNR over the first `N_IMAGES` test faces.  Reported reconstruction: $D_\varepsilon(x_K)$ for the
annealed RED algorithms, $x_K$ otherwise.

In [ ]:
ffhq_test = faces[:N_IMAGES]
table_ffhq = {
    "inpainting":  run_all(hole, ffhq_test, hole_annealed, hole_constant, gs_diffunet, prox_diffunet, L_g_gs_diffunet),
    "demosaicing": run_all(demosaic, ffhq_test, demosaic_annealed, demosaic_constant, gs_diffunet, prox_diffunet, L_g_gs_diffunet),
    "SR x4":       run_all(sr_ffhq, ffhq_test, sr_ffhq_annealed, sr_ffhq_constant, gs_diffunet, prox_diffunet, L_g_gs_diffunet),
}
print(figures.psnr_table(table_ffhq))

## Appendix — CBSD68 (natural images)

Same experiments with the two DRUNet denoisers and $\sigma_0 = 1$: $50\%$ random missing pixels,
sparse-view tomography (60 angles) and super-resolution $\times 4$.

In [ ]:
cbsd68 = data.cbsd68(os.path.join(DATA, "CBSD68_png"), DEVICE, n=68, size=IMAGE_SIZE)
gs_drunet = denoisers.gs_drunet(os.path.join(CKPTS, "gs_drunet.pt"), DEVICE)
prox_drunet = denoisers.prox_drunet(os.path.join(CKPTS, "prox_drunet.pt"), DEVICE, delta=delta)
print(cbsd68.shape)

### Figure — CBSD68, random inpainting ($50\%$ of the pixels)

In [ ]:
# parameters: one line per run
K = 500
sigma_0 = sigma_0_cbsd68
r = 0.98

inpainting_annealed = {
    "red_gd":  Params(K, sigma_0=sigma_0, r=r, eps=0.01, lam=0.3),
    "pnp_pgd": Params(K, sigma_0=sigma_0, r=r, eps=0.02, c_f=0.95),
    "snore":   Params(K, sigma_0=sigma_0, r=r, eps=0.01, lam=0.3, q=0.55),
    "snopnp":  Params(K, sigma_0=sigma_0, r=r, eps=0.01, c_f=0.7, q=1),
    "ered":    Params(K, sigma_0=sigma_0, r=r, eps=0.01, lam=0.3, q=1, warm_mult=1, group="dihedral"),
}
inpainting_constant = {
    "red_gd":  Params(K, sigma_c=0.03, lam=0.5),
    "pnp_pgd": Params(K, sigma_c=0.02, c_f=0.95),
    "snore":   Params(K, sigma_c=0.06, lam=0.3, q=0.7),
    "snopnp":  Params(K, sigma_c=0.03, c_f=0.95, q=1),
    "ered":    Params(K, sigma_c=0.03, lam=0.5, q=0.55, group="dihedral"),
}

In [ ]:
inpainting = problems.random_inpainting(IMAGE_SIZE, DEVICE, missing_ratio=0.5, sigma_noise=0.01)
_ = paper_figure(inpainting, cbsd68, 65, inpainting_annealed, inpainting_constant, gs_drunet, prox_drunet, L_g_gs_drunet)

### Figure — CBSD68, sparse-view tomography (60 angles)

The longest runs of the paper: $K = 1000$.

In [ ]:
# parameters: one line per run
K = 1000
sigma_0 = sigma_0_cbsd68
r = 0.98

tomo_annealed = {
    "red_gd":  Params(K, sigma_0=sigma_0, r=r, eps=0.028, lam=0.03),
    "pnp_pgd": Params(K, sigma_0=sigma_0, r=r, eps=0.01, c_f=0.95),
    "snore":   Params(K, sigma_0=sigma_0, r=r, eps=0.04, lam=0.03, q=0.55, warm_mult=0.1),
    "snopnp":  Params(K, sigma_0=sigma_0, r=r, eps=0.01, c_f=0.7, q=0.7, tau="coupled"),
    "ered":    Params(K, sigma_0=sigma_0, r=r, eps=0.028, lam=0.03, q=0.55, warm_mult=0.1, group="dihedral"),
}
tomo_constant = {
    "red_gd":  Params(K, sigma_c=0.0571, lam=0.03),
    "pnp_pgd": Params(K, sigma_c=0.04, c_f=0.95),
    "snore":   Params(K, sigma_c=0.08, lam=0.03, q=0.55),
    "snopnp":  Params(K, sigma_c=0.0571, c_f=0.95, q=0.7, tau="coupled"),
    "ered":    Params(K, sigma_c=0.0571, lam=0.03, q=0.55, group="dihedral"),
}

In [ ]:
tomo = problems.tomography(IMAGE_SIZE, DEVICE, angles=60, sigma_noise=0.01)
_ = paper_figure(tomo, cbsd68, 31, tomo_annealed, tomo_constant, gs_drunet, prox_drunet, L_g_gs_drunet)

### CBSD68, super-resolution $\times 4$ (table only)

In [ ]:
# parameters: one line per run
K = 500
sigma_0 = sigma_0_cbsd68
r = 0.98

sr_cbsd68_annealed = {
    "red_gd":  Params(K, sigma_0=sigma_0, r=r, eps=0.02, lam=0.1),
    "pnp_pgd": Params(K, sigma_0=sigma_0, r=r, eps=0.028, c_f=0.7),
    "snore":   Params(K, sigma_0=sigma_0, r=r, eps=0.04, lam=0.03, q=1, warm_mult=0.1),
    "snopnp":  Params(K, sigma_0=sigma_0, r=r, eps=0.02, c_f=0.5, q=1, warm_mult=0.1),
    "ered":    Params(K, sigma_0=sigma_0, r=r, eps=0.02, lam=0.1, q=0.55, warm_mult=1, group="dihedral"),
}
sr_cbsd68_constant = {
    "red_gd":  Params(K, sigma_c=0.04, lam=0.06),
    "pnp_pgd": Params(K, sigma_c=0.04, c_f=0.95),
    "snore":   Params(K, sigma_c=0.056, lam=0.06, q=0.55),
    "snopnp":  Params(K, sigma_c=0.04, c_f=0.95, q=1),
    "ered":    Params(K, sigma_c=0.04, lam=0.06, q=0.55, group="dihedral"),
}

### Table 2 — CBSD68

In [ ]:
sr_cbsd68 = problems.super_resolution(IMAGE_SIZE, DEVICE, factor=4, sigma_noise=0.01)
cbsd68_test = cbsd68[:N_IMAGES]
table_cbsd68 = {
    "inpainting": run_all(inpainting, cbsd68_test, inpainting_annealed, inpainting_constant, gs_drunet, prox_drunet, L_g_gs_drunet),
    "tomography": run_all(tomo, cbsd68_test, tomo_annealed, tomo_constant, gs_drunet, prox_drunet, L_g_gs_drunet),
    "SR x4":      run_all(sr_cbsd68, cbsd68_test, sr_cbsd68_annealed, sr_cbsd68_constant, gs_drunet, prox_drunet, L_g_gs_drunet),
}
print(figures.psnr_table(table_cbsd68))

## Changing something

* another problem: `problems.hole_inpainting(128, DEVICE, hole=40)`, then your own `Params` lines;
* another schedule: pass your own array of noise levels to any function of `algorithms.py`;
* another denoiser: wrap any network `N(x, sigma)` in `denoisers.GradientStepDenoiser`.